In [18]:
import urllib.request
from datetime import datetime
import os
import pandas as pd
import re

In [24]:
province_id = 1

while province_id < 28:
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"province_{province_id}_{timestamp}.csv"
    filepath = os.path.join("data", filename)
    # перевіряємо, чи вже є файл з таким ім’ям
    filename_base = f"province_{province_id}"
    files = os.listdir("data")

    found_substring = False
    for s in files:
        if filename_base in s:
            found_substring = True
            break  

    if not found_substring:
        urllib.request.urlretrieve(url, filepath)
        print(f"Завантажено: {filename}")
    else:
        print(f"Файл {filename} вже існує. Пропускаємо...")
    province_id += 1
    

Файл province_1_20250925_171817.csv вже існує. Пропускаємо...
Файл province_2_20250925_171817.csv вже існує. Пропускаємо...
Файл province_3_20250925_171817.csv вже існує. Пропускаємо...
Файл province_4_20250925_171817.csv вже існує. Пропускаємо...
Файл province_5_20250925_171817.csv вже існує. Пропускаємо...
Файл province_6_20250925_171817.csv вже існує. Пропускаємо...
Файл province_7_20250925_171817.csv вже існує. Пропускаємо...
Файл province_8_20250925_171817.csv вже існує. Пропускаємо...
Файл province_9_20250925_171817.csv вже існує. Пропускаємо...
Файл province_10_20250925_171817.csv вже існує. Пропускаємо...
Файл province_11_20250925_171817.csv вже існує. Пропускаємо...
Файл province_12_20250925_171817.csv вже існує. Пропускаємо...
Файл province_13_20250925_171817.csv вже існує. Пропускаємо...
Файл province_14_20250925_171817.csv вже існує. Пропускаємо...
Файл province_15_20250925_171817.csv вже існує. Пропускаємо...
Файл province_16_20250925_171817.csv вже існує. Пропускаємо...
Ф

In [ ]:
import pandas as pd
import re

def remove_html_tags(text):
    return re.sub(r'<.*?>', '', text)

# 1. Вказуємо шлях до папки з CSV-файлами
folder_path = "data"

# 2. Зчитуємо усі CSV-файли у список DataFrame
all_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

df_list = []
for file in all_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path, skiprows=1)

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].apply(remove_html_tags)
    # --- Data Cleaning ---
    columns_to_keep = ['year','week','SMN','SMT','VCI','TCI','VHI']
    df.columns = [remove_html_tags(str(col)).strip() for col in df.columns]
    available_columns = [col for col in columns_to_keep if col in df.columns]
    df = df[available_columns]
    
    df = df.fillna(0)
    
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
        remove_html_tags(df[col])
    
    # Додаємо стовпець з індексом області
    # Витягуємо province_id з імені файлу
    match = re.search(r'province_(\d+)_', file)
    if match:
        province_id = int(match.group(1))
        df['province_id'] = province_id
    else:
        df['province_id'] = -1  # якщо не знайдено
    
    df_list.append(df)

# 3. Об’єднуємо всі DataFrame в один
final_df = pd.concat(df_list, ignore_index=True)
print (final_df.head(10))
# TODO: Fix names to english
province_names = {
    1: "Вінницька",
    2: "Волинська",
    3: "Дніпропетровська",
    4: "Донецька",
    5: "Житомирська",
    6: "Закарпатська",
    7: "Запорізька",
    8: "Івано-Франківська",
    9: "Київська",
    10: "Кіровоградська",
    11: "Луганська",
    12: "Львівська",
    13: "Миколаївська",
    14: "Одеська",
    15: "Полтавська",
    16: "Рівненська",
    17: "Сумська",
    18: "Тернопільська",
    19: "Харківська",
    20: "Херсонська",
    21: "Хмельницька",
    22: "Черкаська",
    23: "Чернівецька",
    24: "Чернігівська",
    25: "Республіка Крим",
    26: "м. Київ",
    27: "Севастополь"
}

final_df["province_name"] = final_df["province_id"].map(province_names)

final_df = final_df[final_df["province_id"] != -1]
final_df = final_df.dropna()
print(final_df.head(30))
    

    

    year   week     SMN    SMT    VCI    TCI  VHI  province_id   province_name
0    1.0  0.059  258.24  51.11  48.78  49.95  0.0           10  Кіровоградська
1    2.0  0.063  261.53  55.89  38.20  47.04  0.0           10  Кіровоградська
2    3.0  0.063  263.45  57.30  32.69  44.99  0.0           10  Кіровоградська
3    4.0  0.061  265.10  53.96  28.62  41.29  0.0           10  Кіровоградська
4    5.0  0.058  266.42  46.87  28.57  37.72  0.0           10  Кіровоградська
5    6.0  0.056  267.47  39.55  30.27  34.91  0.0           10  Кіровоградська
6    7.0  0.055  268.58  35.19  31.10  33.14  0.0           10  Кіровоградська
7    8.0  0.057  270.15  33.35  32.09  32.72  0.0           10  Кіровоградська
8    9.0  0.057  271.60  30.82  34.71  32.77  0.0           10  Кіровоградська
9   10.0  0.057  273.10  27.66  36.79  32.23  0.0           10  Кіровоградська
10  11.0  0.063  275.28  26.28  34.48  30.38  0.0           10  Кіровоградська
11  12.0  0.074  277.68  25.86  36.39  31.12  0.0   